Retrain SVC models for plots

# Libraries

In [1]:
import os
import shutil
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
RANDOM_STATE = 42

In [3]:
# train/test split
import sys
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [4]:
# dict with num features
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import dict_num_features

# `probability=True` in SVC

In [5]:
df_metrics = pd.read_excel(r'../results/metrics_modelling2_filtered_optuna.xlsx', index_col=[0])

In [6]:
data_mask = df_metrics['dataset'].isin(
    ['df_modelling_dimensionless', 'df_modelling_dimensionless_pf']
)
svc_mask = df_metrics['model'].str.contains('svc')

splashing_mask = df_metrics['target'] == 'splashing'
net_impact_mask = df_metrics['target'] == 'net_impact'

df_splashing = df_metrics[data_mask & svc_mask & splashing_mask].sort_values(
    ['roc_auc', 'f1'], ascending=False
)

df_net_impact = df_metrics[data_mask & svc_mask & net_impact_mask].sort_values(
    ['roc_auc', 'f1'], ascending=False
)

print('splashing')
display(df_splashing)

print('net_impact')
display(df_net_impact)

splashing


,dataset,target,model,accuracy,f1,precision,recall,roc_auc,optuna_flg
59,df_modelling_dimensionless,splashing,svc_smote_splashing_ordenc,0.920000,0.937500,0.937500,0.937500,0.913194,1.0
55,df_modelling_dimensionless,splashing,svc_smote_splashing_onehot,0.920000,0.938776,0.920000,0.958333,0.905093,1.0
56,df_modelling_dimensionless_pf,splashing,svc_smote_splashing_onehot,0.920000,0.938776,0.920000,0.958333,0.905093,1.0
23,df_modelling_dimensionless,splashing,svc_smote_splashing_onehot,0.853333,0.888889,0.862745,0.916667,0.828704,NaN
24,df_modelling_dimensionless_pf,splashing,svc_smote_splashing_onehot,0.853333,0.888889,0.862745,0.916667,0.828704,NaN
27,df_modelling_dimensionless,splashing,svc_smote_splashing_ordenc,0.840000,0.877551,0.860000,0.895833,0.818287,NaN


net_impact


,dataset,target,model,accuracy,f1,precision,recall,roc_auc,optuna_flg
46,df_modelling_dimensionless_pf,net_impact,svc_smote_net_impact_ordenc,0.946667,0.900000,0.900000,0.90,0.931818,1.0
45,df_modelling_dimensionless,net_impact,svc_smote_net_impact_ordenc,0.933333,0.871795,0.894737,0.85,0.906818,1.0
13,df_modelling_dimensionless,net_impact,svc_smote_net_impact_ordenc,0.880000,0.800000,0.720000,0.90,0.886364,NaN
14,df_modelling_dimensionless_pf,net_impact,svc_smote_net_impact_ordenc,0.880000,0.800000,0.720000,0.90,0.886364,NaN


- best_models_modelling_2/svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact.pkl

# Net impact

In [7]:
path_pipeline = r'../results/best_models_modelling_2/svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact.pkl'
pipeline = joblib.load(path_pipeline)
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Re^(1/4)', 'We^(1/4)^2',
                                                   'We^(1/4)^2 Re^(1/4)',
                                                   'inclination',
                                                   'particle_liquid_density_ratio',
                                                   'particle_droplet_diameter_ratio']),
                                                 ('cat',
                                                  Pipeline(steps=[('ordenc',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=nan))]),
                                                  ['wettability'])])),
                ('model',
                 SVC(C=4026.5502671808963, coef0=-0.17141513324034952,
                     gamma=1.2146813942954187))])

In [8]:
pipeline.steps[1][1].probability = True
pipeline.steps[1][1].get_params()['probability']

True

## Chossing corresponding data

In [9]:
# svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact
row_model_data = df_metrics.loc[
    (df_metrics['target']=='net_impact') &
    (df_metrics['dataset']=='df_modelling_dimensionless_pf') &
    (df_metrics['model']=='svc_smote_net_impact_ordenc') &
    (df_metrics['optuna_flg']==1)
].iloc[0]
row_model_data

dataset       df_modelling_dimensionless_pf
target                           net_impact
model           svc_smote_net_impact_ordenc
accuracy                           0.946667
f1                                      0.9
precision                               0.9
recall                                  0.9
roc_auc                            0.931818
optuna_flg                              1.0
Name: 46, dtype: object

In [10]:
target = 'net_impact'
train, test = get_train_test(
    dataset_filename=row_model_data['dataset'],
    target=target)
train = train[dict_num_features[row_model_data['dataset']]+['wettability']+[target]]
test = test[dict_num_features[row_model_data['dataset']]+['wettability']+[target]]

## Fitting pipeline and saving

In [11]:
pipeline.fit(X=train.drop(columns=[target]), y=train[target])
display(pipeline.predict_proba(test.drop(columns=[target]))[:10])

array([[0.9641    , 0.0359    ],
       [0.78270706, 0.21729294],
       [0.76827281, 0.23172719],
       [0.99414614, 0.00585386],
       [0.96457008, 0.03542992],
       [0.7855511 , 0.2144489 ],
       [0.04103175, 0.95896825],
       [0.78206326, 0.21793674],
       [0.76941575, 0.23058425],
       [0.7818408 , 0.2181592 ]])

Check that metrics are the same

In [12]:
preds = pipeline.predict(test.drop(columns=[target]))
metrics = [
    {
        'accuracy': accuracy_score(test[target], preds),
        'f1': f1_score(test[target], preds),
        'precision': precision_score(test[target], preds),
        'recall': recall_score(test[target], preds),
        'roc_auc': roc_auc_score(test[target], preds)
    }
]
pd.DataFrame(metrics).T

,0
accuracy,0.946667
f1,0.900000
precision,0.900000
recall,0.900000
roc_auc,0.931818


In [13]:
path_prob_models = Path('../results/probability_models')
if not os.path.exists(path_prob_models): os.makedirs(path_prob_models)

pipeline_name = (
    'svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact'
)
joblib.dump(pipeline, str(path_prob_models / f'{pipeline_name}.pkl'))

['../results/probability_models/svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact.pkl']

# Splashing

In [14]:
path_pipeline = r'../results/best_models_modelling_2/svc_smote_splashing_ordenc_df_modelling_dimensionless_splashing.pkl'
pipeline = joblib.load(path_pipeline)
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Re', 'We', 'We_Re',
                                                   'inclination',
                                                   'particle_liquid_density_ratio',
                                                   'particle_droplet_diameter_ratio']),
                                                 ('cat',
                                                  Pipeline(steps=[('ordenc',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=nan))]),
                                                  ['wettability'])])),
                ('model',
                 SVC(C=253.4525127339649, coef0=-0.05553622992136159,
                     gamma=0.03430692477759674))])

In [15]:
pipeline.steps[1][1].probability = True
pipeline.steps[1][1].get_params()['probability']

True

## Chossing corresponding data

In [16]:
# svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact
row_model_data = df_metrics.loc[
    (df_metrics['target']=='splashing') &
    (df_metrics['dataset']=='df_modelling_dimensionless') &
    (df_metrics['model']=='svc_smote_splashing_ordenc') &
    (df_metrics['optuna_flg']==1)
].iloc[0]
row_model_data

dataset       df_modelling_dimensionless
target                         splashing
model         svc_smote_splashing_ordenc
accuracy                            0.92
f1                                0.9375
precision                         0.9375
recall                            0.9375
roc_auc                         0.913194
optuna_flg                           1.0
Name: 59, dtype: object

In [17]:
target = 'splashing'
train, test = get_train_test(
    dataset_filename=row_model_data['dataset'],
    target=target)
train = train[dict_num_features[row_model_data['dataset']]+['wettability']+[target]]
test = test[dict_num_features[row_model_data['dataset']]+['wettability']+[target]]

## Fitting pipeline and saving

In [18]:
pipeline.fit(X=train.drop(columns=[target]), y=train[target])
display(pipeline.predict_proba(test.drop(columns=[target]))[:10])

array([[0.04209124, 0.95790876],
       [0.56642668, 0.43357332],
       [0.15918569, 0.84081431],
       [0.93793917, 0.06206083],
       [0.08430777, 0.91569223],
       [0.80506149, 0.19493851],
       [0.10312642, 0.89687358],
       [0.12625378, 0.87374622],
       [0.35207483, 0.64792517],
       [0.77464894, 0.22535106]])

Check that metrics are the same

In [19]:
preds = pipeline.predict(test.drop(columns=[target]))
metrics = [
    {
        'accuracy': accuracy_score(test[target], preds),
        'f1': f1_score(test[target], preds),
        'precision': precision_score(test[target], preds),
        'recall': recall_score(test[target], preds),
        'roc_auc': roc_auc_score(test[target], preds)
    }
]
pd.DataFrame(metrics).T

,0
accuracy,0.920000
f1,0.937500
precision,0.937500
recall,0.937500
roc_auc,0.913194


In [20]:
path_prob_models = Path('../results/probability_models')
if not os.path.exists(path_prob_models): os.makedirs(path_prob_models)

pipeline_name = (
    'svc_smote_splashing_ordenc_df_modelling_dimensionless_splashing'
)
joblib.dump(pipeline, str(path_prob_models / f'{pipeline_name}.pkl'))

['../results/probability_models/svc_smote_splashing_ordenc_df_modelling_dimensionless_splashing.pkl']